In [86]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [87]:
path =  '/content/drive/MyDrive/KhaiPhaDL/BTL/'

In [88]:
import pandas as pd

In [89]:
import re


In [90]:
!pip install transformers torch pandas scikit-learn

# Bảng mô tả dataset

| Tên cột                | Giải thích chuẩn                                                                                                                                                                                                          |
| ---------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `product_id`           | Mã định danh sản phẩm. Dùng để xác định đánh giá thuộc về sản phẩm nào. Không phản ánh trực tiếp cảm xúc hay nội dung đánh giá.                                                                                           |
| `review_id`            | Mã định danh duy nhất của từng đánh giá. Đây là khóa nhận diện bản ghi, không có ý nghĩa dự đoán.                                                                                                                         |
| `customer_name`        | Tên khách hàng thực hiện đánh giá. Cột này không liên quan trực tiếp đến việc phân loại cảm xúc theo khía cạnh sản phẩm.                                                                                                  |
| `rating`               | Điểm đánh giá tổng quát của khách hàng, thường từ 1 đến 5 sao. Cột này phản ánh mức độ hài lòng chung của khách hàng về sản phẩm.                                                                                         |
| `title`                | Tiêu đề ngắn của đánh giá. Có thể chứa cảm xúc tổng quan như “Cực kì hài lòng”, “Không hài lòng”, “Tạm ổn”.                                                                                                               |
| `content`              | Nội dung đánh giá chi tiết của khách hàng. Đây là nguồn thông tin chính để nhận diện khách hàng đang nói đến khía cạnh nào và cảm xúc tương ứng là gì.                                                                    |
| `purchased`            | Thông tin cho biết khách hàng đã mua sản phẩm hay chưa. Có thể dùng để kiểm tra độ tin cậy của đánh giá.                                                                                                                  |
| `created_at`           | Thời điểm đánh giá được tạo. Có thể dùng để phân tích xu hướng đánh giá theo thời gian.                                                                                                                                   |
| `nhan_chat_lieu`       | Nhãn cảm xúc cho khía cạnh chất liệu. Gán nhãn khi nội dung nhắc đến loại vải, cảm giác bề mặt, độ mỏng/dày, mềm/cứng, nóng/mát, co giãn hoặc thấm hút mồ hôi.                                                            |
| `nhan_kich_co_form`    | Nhãn cảm xúc cho khía cạnh kích cỡ và trải nghiệm mặc. Gán nhãn khi nội dung nhắc đến size, chiều cao, cân nặng, số đo, form rộng/chật/ngắn/dài, cảm giác mặc vừa, kích, thùng thình hoặc tôn dáng.                       |
| `nhan_thiet_ke`        | Nhãn cảm xúc cho khía cạnh thiết kế và thẩm mỹ. Gán nhãn khi nội dung nhắc đến vẻ đẹp bên ngoài, màu sắc, hoa văn, họa tiết, độ xòe, sự trẻ trung, sang trọng, già dặn hoặc quê mùa.                                      |
| `nhan_gia_cong`        | Nhãn cảm xúc cho khía cạnh kỹ thuật gia công và phụ kiện. Gán nhãn khi nội dung nhắc đến đường kim, mũi chỉ, chỉ thừa, vắt sổ, nếp gấp, dây đai, dây thắt lưng, khóa kéo, nút cài hoặc khuy áo.                           |
| `nhan_gia_tri_thuc_te` | Nhãn cảm xúc cho khía cạnh giá trị và độ trung thực. Gán nhãn khi nội dung nhắc đến giá tiền, sản phẩm có đáng tiền không, có giống hình quảng cáo không, hàng nhận được có đúng mô tả không hoặc mức độ uy tín của shop. |


## Mục tiêu bài toán

Mục tiêu của bài toán là xây dựng mô hình **phân tích cảm xúc theo khía cạnh** cho các đánh giá sản phẩm quần áo.

Dữ liệu đầu vào bao gồm:

- `rating`: điểm đánh giá của khách hàng
- `content`: nội dung bình luận của khách hàng

Mô hình cần tự động nhận diện và phân loại cảm xúc đối với từng khía cạnh của sản phẩm, bao gồm:

| Khía cạnh | Ý nghĩa |
|---|---|
| Chất liệu | Đánh giá về vải, độ mềm, độ dày, độ mát, độ co giãn, cảm giác khi mặc |
| Kích cỡ / form dáng | Đánh giá về size, độ vừa vặn, form rộng/chật/ngắn/dài, trải nghiệm mặc |
| Thiết kế | Đánh giá về màu sắc, kiểu dáng, họa tiết, tính thẩm mỹ và thời trang |
| Gia công | Đánh giá về đường may, chỉ thừa, khóa kéo, nút áo, phụ kiện đi kèm |

Mỗi khía cạnh được gán một trong các nhãn cảm xúc sau:

| Nhãn | Ý nghĩa |
|---|---|
| `Positive` | Tích cực |
| `Negative` | Tiêu cực |
| `Neutral` | Trung tính |
| `None` | Không đề cập |

Kết quả của bài toán giúp doanh nghiệp khai thác phản hồi khách hàng một cách chi tiết hơn, từ đó xác định điểm mạnh, điểm yếu của sản phẩm, cải thiện chất lượng quần áo, tối ưu thiết kế và nâng cao trải nghiệm người dùng.

# Tiền xử lí dữ liệu




In [ ]:
df = pd.read_csv(path + 'Dataset_quan_relabeled.csv')

In [92]:
df.head(5)

,product_id,review_id,customer_name,rating,title,content,purchased,created_at,nhan_chat_lieu,nhan_kich_co_form,nhan_thiet_ke,nhan_gia_cong,nhan_gia_tri_thuc_te
0,145290254,15592989,Anh Chi,5,Cực kì hài lòng,Chất lượng ok- from vừa vặn- đường may tỉ mỉ- ...,NaN,1648191789,NaN,Positive,NaN,Positive,Positive
1,188380252,16883252,Hồng Thắng,5,Cực kì hài lòng,quá đẹp,NaN,1656497043,NaN,NaN,NaN,NaN,NaN
2,188380252,16888359,Trương Hồng,5,Cực kì hài lòng,Hang dep . May ky chat lieu vai ok . Se ung h...,NaN,1656551226,NaN,Positive,NaN,Positive,Positive
3,14815891,3550598,Tô Thị Hằng,1,Rất không hài lòng,Mình mua chân váy này vì nhầm size mình đã trả...,NaN,1589340963,NaN,NaN,NaN,NaN,Negative
4,14815891,10012186,Nguyễn Thị Hồng Nga,5,Cực kì hài lòng,rất hài lòng,NaN,1622187534,NaN,NaN,NaN,NaN,NaN


##Xử lí trùng lặp

In [93]:
# Kiểm tra số lượng giá trị rỗng theo từng cột trong df
missing_values = df.isnull().sum()
print("Số lượng giá trị rỗng theo từng cột:")
display(missing_values)

Số lượng giá trị rỗng theo từng cột:


,0
product_id,0
review_id,0
customer_name,609
rating,0
title,1
content,0
purchased,6450
created_at,0
nhan_chat_lieu,4747
nhan_kich_co_form,4826


In [94]:
df.dropna(subset=['content'], inplace=True)

In [95]:
df.shape

(6450, 13)

In [96]:
# Kiểm tra số lượng giá trị rỗng theo từng cột trong df
missing_values = df.isnull().sum()
print("Số lượng giá trị rỗng theo từng cột:")
display(missing_values)

Số lượng giá trị rỗng theo từng cột:


,0
product_id,0
review_id,0
customer_name,609
rating,0
title,1
content,0
purchased,6450
created_at,0
nhan_chat_lieu,4747
nhan_kich_co_form,4826


In [97]:
def handle_duplicates_and_missing(df):
    # 1. Xóa các dòng trùng lặp dựa trên review_id
    df = df.drop_duplicates(subset=['review_id']).copy()

    # 2. Xóa bỏ cột 'purchased' do phần lớn là rỗng
    if 'purchased' in df.columns:
        df = df.drop(columns=['purchased'])

    return df


In [98]:
# Chạy hàm
df = handle_duplicates_and_missing(df)

In [99]:
df.shape

(6142, 12)

In [100]:
# Kiểm tra số lượng giá trị rỗng theo từng cột trong df
missing_values = df.isnull().sum()
print("Số lượng giá trị rỗng theo từng cột:")
display(missing_values)

Số lượng giá trị rỗng theo từng cột:


,0
product_id,0
review_id,0
customer_name,570
rating,0
title,1
content,0
created_at,0
nhan_chat_lieu,4535
nhan_kich_co_form,4613
nhan_thiet_ke,5150


In [101]:
df.head()

,product_id,review_id,customer_name,rating,title,content,created_at,nhan_chat_lieu,nhan_kich_co_form,nhan_thiet_ke,nhan_gia_cong,nhan_gia_tri_thuc_te
0,145290254,15592989,Anh Chi,5,Cực kì hài lòng,Chất lượng ok- from vừa vặn- đường may tỉ mỉ- ...,1648191789,NaN,Positive,NaN,Positive,Positive
1,188380252,16883252,Hồng Thắng,5,Cực kì hài lòng,quá đẹp,1656497043,NaN,NaN,NaN,NaN,NaN
2,188380252,16888359,Trương Hồng,5,Cực kì hài lòng,Hang dep . May ky chat lieu vai ok . Se ung h...,1656551226,NaN,Positive,NaN,Positive,Positive
3,14815891,3550598,Tô Thị Hằng,1,Rất không hài lòng,Mình mua chân váy này vì nhầm size mình đã trả...,1589340963,NaN,NaN,NaN,NaN,Negative
4,14815891,10012186,Nguyễn Thị Hồng Nga,5,Cực kì hài lòng,rất hài lòng,1622187534,NaN,NaN,NaN,NaN,NaN


In [102]:
aspect_cols = ['nhan_chat_lieu', 'nhan_kich_co_form', 'nhan_thiet_ke', 'nhan_gia_cong', 'nhan_gia_tri_thuc_te']

for col in aspect_cols:
    print(f"--- Thống kê cột: {col} ---")
    print(df[col].value_counts(dropna=False))
    print("\n")

--- Thống kê cột: nhan_chat_lieu ---
nhan_chat_lieu
NaN         4535
Positive     919
Negative     395
Neutral      293
Name: count, dtype: int64


--- Thống kê cột: nhan_kich_co_form ---
nhan_kich_co_form
NaN         4613
Neutral      647
Positive     577
Negative     305
Name: count, dtype: int64


--- Thống kê cột: nhan_thiet_ke ---
nhan_thiet_ke
NaN         5150
Positive     440
Neutral      355
Negative     197
Name: count, dtype: int64


--- Thống kê cột: nhan_gia_cong ---
nhan_gia_cong
NaN         5538
Neutral      238
Positive     196
Negative     170
Name: count, dtype: int64


--- Thống kê cột: nhan_gia_tri_thuc_te ---
nhan_gia_tri_thuc_te
NaN         2871
Positive    2230
Negative     617
Neutral      424
Name: count, dtype: int64




In [103]:

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6142 entries, 0 to 6449
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   product_id            6142 non-null   int64 
 1   review_id             6142 non-null   int64 
 2   customer_name         5572 non-null   object
 3   rating                6142 non-null   int64 
 4   title                 6141 non-null   object
 5   content               6142 non-null   object
 6   created_at            6142 non-null   int64 
 7   nhan_chat_lieu        1607 non-null   object
 8   nhan_kich_co_form     1529 non-null   object
 9   nhan_thiet_ke         992 non-null    object
 10  nhan_gia_cong         604 non-null    object
 11  nhan_gia_tri_thuc_te  3271 non-null   object
dtypes: int64(4), object(8)
memory usage: 623.8+ KB


## Xóa cột k cần thiết

In [104]:
columns_to_keep = [
    'rating',
    'content',
    'nhan_chat_lieu',
    'nhan_kich_co_form',
    'nhan_thiet_ke',
    'nhan_gia_cong',
    'nhan_gia_tri_thuc_te'
]

df = df[columns_to_keep].copy()

print(f"Số cột sau khi loại bỏ: {len(df.columns)}")
df.shape

Số cột sau khi loại bỏ: 7


(6142, 7)

In [105]:
df.head()

,rating,content,nhan_chat_lieu,nhan_kich_co_form,nhan_thiet_ke,nhan_gia_cong,nhan_gia_tri_thuc_te
0,5,Chất lượng ok- from vừa vặn- đường may tỉ mỉ- ...,NaN,Positive,NaN,Positive,Positive
1,5,quá đẹp,NaN,NaN,NaN,NaN,NaN
2,5,Hang dep . May ky chat lieu vai ok . Se ung h...,NaN,Positive,NaN,Positive,Positive
3,1,Mình mua chân váy này vì nhầm size mình đã trả...,NaN,NaN,NaN,NaN,Negative
4,5,rất hài lòng,NaN,NaN,NaN,NaN,NaN


## Xử lí vấn đề độ dài câu đánh giá

In [106]:
def check_length_extremes(df, column):
    lengths = df[column].astype(str).str.len()
    idx_min = lengths.idxmin()
    idx_max = lengths.idxmax()

    print(f"--- CỘT: {column.upper()} ---")
    print(f"Câu ngắn nhất ({lengths[idx_min]} ký tự):\n\"{df.loc[idx_min, column]}\"")
    print(f"Câu dài nhất ({lengths[idx_max]} ký tự):\n\"{df.loc[idx_max, column]}\"")
    print("\n")

check_length_extremes(df, 'content')

--- CỘT: CONTENT ---
Câu ngắn nhất (1 ký tự):
"h"
Câu dài nhất (1373 ký tự):
"Tôi nói thật đọc xong cuốn truyện này tôi chuyển qua yêu ông HECTOR MALOT luôn. Yêu mến quá nhiều tôi chuyển sang mua hết các tp ông đã viết chỉ vì cuốn " Không gia đình này đây. Các bạn chưa hiểu hết cái niềm xúc động của tôi không khi đang cảm nhận nó. Phải thức mấy đêm liền chỉ để không muốn chậm trễ thấu hiểu hết ý nghĩa câu chuyện. Khi đọc tôi cứ tưởng mình là một người xem phim trong rạp chiếu với lối viết hết sức gần gũi khiến ta bùi ngùi khi đọc phải những tình tiết ngoặt nghoèo bi kịch. Câu chuyện về cậu bé Rémi cùng các bạn đời trên đường thiên lí. Đúng là tình yêu, tình bạn - điều kì diệu bởi sự hiện hữu của nó khiến cuộc đời một cậu bé bất hạnh không còn những khoảng trống của sự mất mát. Ở đời phải hiểu được quy luật của sinh tử, người bên cạnh ta không thể ở bên ta mãi 

In [107]:
# Định nghĩa số lượng từ tối thiểu để được giữ lại
MIN_WORDS = 6

print(f"Số dòng trước khi lọc: {len(df)}")

# Tính số lượng từ của từng dòng trong cột 'content'
# (Dùng hàm split() để tách câu thành các từ dựa trên khoảng trắng)
df['word_count'] = df['content'].apply(lambda x: len(str(x).split()))

# Chỉ giữ lại những hàng có số từ >= MIN_WORDS
df = df[df['word_count'] >= MIN_WORDS].copy()

# Xóa cột phụ 'word_count' đi để bảng dữ liệu gọn gàng như cũ
df = df.drop(columns=['word_count'])

print(f"Số dòng còn lại sau khi bỏ các bình luận quá ngắn: {len(df)}")


Số dòng trước khi lọc: 6142
Số dòng còn lại sau khi bỏ các bình luận quá ngắn: 4339


In [108]:
check_length_extremes(df, 'content')

--- CỘT: CONTENT ---
Câu ngắn nhất (19 ký tự):
"đồ ok, ko bị gì hết"
Câu dài nhất (1373 ký tự):
"Tôi nói thật đọc xong cuốn truyện này tôi chuyển qua yêu ông HECTOR MALOT luôn. Yêu mến quá nhiều tôi chuyển sang mua hết các tp ông đã viết chỉ vì cuốn " Không gia đình này đây. Các bạn chưa hiểu hết cái niềm xúc động của tôi không khi đang cảm nhận nó. Phải thức mấy đêm liền chỉ để không muốn chậm trễ thấu hiểu hết ý nghĩa câu chuyện. Khi đọc tôi cứ tưởng mình là một người xem phim trong rạp chiếu với lối viết hết sức gần gũi khiến ta bùi ngùi khi đọc phải những tình tiết ngoặt nghoèo bi kịch. Câu chuyện về cậu bé Rémi cùng các bạn đời trên đường thiên lí. Đúng là tình yêu, tình bạn - điều kì diệu bởi sự hiện hữu của nó khiến cuộc đời một cậu bé bất hạnh không còn những khoảng trống của sự mất mát. Ở đời phải hiểu được quy luật của sinh tử, người bên cạnh ta không t

In [109]:
# Sắp xếp các câu trong cột 'content' theo độ dài tăng dần
# Tạo một bản sao tạm thời có cột 'length' để dễ quan sát
content_sorted = df[['content']].copy()
content_sorted['length'] = content_sorted['content'].astype(str).str.len()

# Hiển thị kết quả sắp xếp từ thấp đến cao
content_sorted.sort_values(by='length', ascending=True).head(10)

,content,length
292,"đồ ok, ko bị gì hết",19
3489,Bị bẩn ở ruột 1 chút,20
4769,sp ok đúng tầm giá !,20
4983,ao bị bung chỉ ở tay,20
5053,Áo đẹp tốt sẽ ủng hộ,20
5561,Áo đẹp vải khá dày .,20
4281,bọc cẩn thận . ưng ý,20
3053,Quá ổn so với tầm giá,21
380,nón đẹp lắm nên mua 😘,21
6232,Áo đẹp lắm nha mọi ng,21


In [110]:
# Sắp xếp các câu trong cột 'content' theo độ dài tăng dần
# Tạo một bản sao tạm thời có cột 'length' để dễ quan sát
content_sorted = df[['content']].copy()
content_sorted['length'] = content_sorted['content'].astype(str).str.len()

# Hiển thị kết quả sắp xếp từ thấp đến cao
content_sorted.sort_values(by='length', ascending=False).head(10)

,content,length
4062,Tôi nói thật đọc xong cuốn truyện này tô...,1373
4079,Tựa đề câu chuyện là Không gia đình nhưng nhữn...,1225
4065,Đây quả nhiên là một tuyệt phẩm. Chuyện kể về ...,1218
4091,Không gia đình là cuốn truyện tôi đã đọc từ kh...,1070
4064,Nhớ không nhầm thì tôi đọc tác phẩm này lần đầ...,1059
3729,trước hết tôi phải công nhận đây là một tác ph...,1012
4086,“Không gia đình” là cuốn truyện khiến tôi hết ...,1008
4004,"""Không gia đình"" kể về cuộc đời lưu lạc của cậ...",966
4095,"""Không gia đình"" xứn đáng là 1 tác phẩm lớn củ...",965
3998,1 câu chuyện dài về nhân vật chính là cậu bé R...,944


## Xử lí vấn đề content không liên quan đến quần áo ( lỗi)

In [111]:
import pandas as pd
import unicodedata
from typing import List, Optional, Tuple

def filter_clothing_reviews(
    df: pd.DataFrame,
    text_col: str = 'content',
    remove_keywords: Optional[List[str]] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hàm lọc dữ liệu review theo từ khóa để loại bỏ rác/sách/nhận xu.
    (Chỉ sử dụng blacklist để tránh xóa nhầm các bình luận chung chung hợp lệ như "Giao hàng nhanh", "Đẹp").
    """

    if remove_keywords is None:
        remove_keywords = [
            'nhận xu', 'lấy xu', 'mang tính chất minh họa', 'mang tính chất minh hoạ',
            'tính chất nhận xu', 'hình ảnh mang', 'kiếm xu', 'để nhận', 'chỉ mang tính',
            'sách', 'quyển', 'trang sách', 'đọc', 'tác giả', 'nội dung sách',
            'truyện', 'tiểu thuyết', 'bookmark', 'fahasa', 'nhà sách', 'nhà xuất bản', 'bìa',
            'tác phẩm', 'nhân vật'
        ]

    df_filtered = df.copy()

    # LỖI KỸ THUẬT TIẾNG VIỆT: Dữ liệu của bạn chứa font chữ bị phân mảnh (Unicode NFD).
    # Nghĩa là chữ "truyện" trong file của bạn khác mã máy tính với chữ "truyện" gõ bình thường.
    # Do đó cần phải dùng thư viện unicodedata để chuẩn hoá lại (NFC) trước khi so sánh.
    def normalize_text(text):
        if pd.isna(text):
            return ""
        # Chuẩn hóa Unicode về dạng NFC (chuẩn Dựng sẵn) rồi mới chuyển sang in thường
        return unicodedata.normalize('NFC', str(text)).lower()

    temp_col = 'temp_lower_text'
    df_filtered[temp_col] = df_filtered[text_col].apply(normalize_text)

    # 1. Loại bỏ các dòng chứa từ khóa rác / không liên quan (blacklist)
    if remove_keywords:
        # Đồng thời cũng phải chuẩn hoá luôn list từ khoá blacklist để đảm bảo 100% giống nhau
        normalized_keywords = [unicodedata.normalize('NFC', kw).lower() for kw in remove_keywords]
        remove_pattern = '|'.join(normalized_keywords)

        mask_remove = df_filtered[temp_col].str.contains(remove_pattern, na=False, regex=True)
        df_removed = df_filtered[mask_remove]
        df_filtered = df_filtered[~mask_remove]
    else:
        df_removed = pd.DataFrame(columns=df_filtered.columns)

    # Xóa cột tạm
    df_filtered = df_filtered.drop(columns=[temp_col])
    if not df_removed.empty:
        df_removed = df_removed.drop(columns=[temp_col])

    return df_filtered, df_removed

if __name__ == "__main__":
    blacklist = [
        'nhận xu', 'lấy xu', 'mang tính chất minh họa', 'mang tính chất minh hoạ',
        'tính chất nhận xu', 'hình ảnh mang', 'kiếm xu', 'để nhận', 'chỉ mang tính',
        'sách', 'quyển', 'trang sách', 'đọc', 'tác giả', 'nội dung sách',
        'truyện', 'tiểu thuyết', 'bookmark', 'fahasa', 'nhà sách', 'nhà xuất bản', 'bìa',
        'tác phẩm', 'nhân vật', 'hector malot', 'Câu chuyện', 'cậu chuyện'
    ]

    df_clean, df_removed = filter_clothing_reviews(
        df=df,
        text_col='content',
        remove_keywords=blacklist
    )

    print(f"Số lượng đánh giá ban đầu: {len(df)}")
    print(f"Số lượng đánh giá sau khi lọc (được giữ lại): {len(df_clean)}")
    print(f"Số lượng đánh giá bị loại bỏ (nhận xu, sách): {len(df_removed)}\n")

    print("--- 20 DÒNG BỊ XÓA (MẪU) ---")
    # Lấy mẫu để in ra, nếu số lượng bị xóa ít hơn 20 thì lấy toàn bộ
    sample_size = min(20, len(df_removed))
    if sample_size > 0:
        sample_removed = df_removed.sample(sample_size)
        for idx, row in sample_removed.iterrows():
            print(f"[Dòng {idx}]: {row['content']}")
            print("-" * 50)
    else:
        print("Không có dòng nào bị xóa!")


Số lượng đánh giá ban đầu: 4339
Số lượng đánh giá sau khi lọc (được giữ lại): 3947
Số lượng đánh giá bị loại bỏ (nhận xu, sách): 392

--- 20 DÒNG BỊ XÓA (MẪU) ---
[Dòng 3635]: Giao hàng nhanh. Bìa đẹp, một số trang bị rách nhưng không đáng kể.
--------------------------------------------------
[Dòng 3410]: Chữ phiên âm TV. Đọc rất khó chịu
--------------------------------------------------
[Dòng 3764]: Đọc để hiểu hơn về thứ đc gọi là gđ và nhà
--------------------------------------------------
[Dòng 3841]: nhiều vấn đề tưởng chừng như rất nhỏ nhưng có ảnh hưởng rất lớn đến chúng ta. Đọc thấy hiểu bản thân và thông cảm cho những người xung quanh nhiều hơn ấy.
--------------------------------------------------
[Dòng 4027]: truyện rất hay ạ,nói về cuộc đời của 1 cậu bé với những tuổi thơ thăng trầm,nội dung ý nghĩa. tiki giao hàng nhanh.
--------------------------------------------------
[Dòng 4081]: Không gia đình là cuốn tiểu thuyết đầu tiên mình đọc và cũng là cuốn truyện thiếu nhi mà

In [112]:
# Sắp xếp các câu trong cột 'content' theo độ dài tăng dần
# Tạo một bản sao tạm thời có cột 'length' để dễ quan sát
content_sorted = df_clean[['content']].copy()
content_sorted['length'] = content_sorted['content'].astype(str).str.len()

# Hiển thị kết quả sắp xếp từ thấp đến cao
content_sorted.sort_values(by='length', ascending=False).head(10)

,content,length
1174,Mới ship tới chiều nay. \n- Váy màu đen nên mặ...,575
3994,Bé Remi bao nhiêu tuổi nhỉ? Thực sự là nhìn cu...,558
5570,"Áo giống hình, vải mát không quá dày cũng ko q...",545
2891,"Chất liệu vải co dãn rất tốt, mặc thoải mái và...",539
2302,Nhận hàng sáng 18/5: Yêu cầu trả hàng do không...,521
2896,"Vừa nhận gói hàng từ tiki, mở ra ưng ý quá nên...",479
2892,"Giao hàng nhanh hơn dự kiến, chất vải sờ rất đ...",469
2527,"Thực sự sản phẩm quá tệ, rất chi là tệ không c...",467
2767,Cho 1 sao mà tôi còn thấy tiki không xứng!...,448
2911,5* cho shop và chất lượng sản phẩm( không phải...,426


In [113]:
df_clean.head(5)

,rating,content,nhan_chat_lieu,nhan_kich_co_form,nhan_thiet_ke,nhan_gia_cong,nhan_gia_tri_thuc_te
0,5,Chất lượng ok- from vừa vặn- đường may tỉ mỉ- ...,NaN,Positive,NaN,Positive,Positive
2,5,Hang dep . May ky chat lieu vai ok . Se ung h...,NaN,Positive,NaN,Positive,Positive
3,1,Mình mua chân váy này vì nhầm size mình đã trả...,NaN,NaN,NaN,NaN,Negative
6,2,"Sản phẩm từ kiểu dáng đến chất lượng đều kém, ...",NaN,Negative,Negative,NaN,Negative
7,5,"Mua 1 cái màu vàng, 1 cái màu tím từ shop. Chấ...",Neutral,Neutral,Positive,NaN,Positive


# Model

In [114]:

from sklearn.model_selection import train_test_split

In [115]:


# 1. Mã hóa Label thành số
label_map = {
    "None": 0,
    "Positive": 1,
    "Negative": 2,
    "Neutral": 3
}
aspect_cols = ['nhan_chat_lieu', 'nhan_kich_co_form', 'nhan_thiet_ke', 'nhan_gia_cong', 'nhan_gia_tri_thuc_te']

# Đảm bảo sử dụng đúng tên biến DataFrame đã lọc ở bước trước (df_clean)
for col in aspect_cols:
    df_clean[col] = df_clean[col].fillna("None").map(label_map)

# 2. Xử lý Input
df_clean['content'] = df_clean['content'].fillna("")
df_clean['model_input'] = "Đánh giá " + df_clean['rating'].astype(str) + " sao. " + df_clean['content']

# 3. Chia tập Train - Val - Test (80 - 10 - 10)
# Chia 80% cho train, 20% còn lại cho temp (val + test)
df_train, df_temp = train_test_split(df_clean, test_size=0.2, random_state=42)

# Chia 20% temp thành 50/50 để có 10% val và 10% test
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

print(f"Số mẫu Train: {len(df_train)}")
print(f"Số mẫu Validation: {len(df_val)}")
print(f"Số mẫu Test: {len(df_test)}")

Số mẫu Train: 3157
Số mẫu Validation: 395
Số mẫu Test: 395


In [116]:
df_train.head()

,rating,content,nhan_chat_lieu,nhan_kich_co_form,nhan_thiet_ke,nhan_gia_cong,nhan_gia_tri_thuc_te,model_input
6192,5,"Mình hài lòng về sản phẩm, mặc mát, có nhiều t...",1,0,3,0,1,"Đánh giá 5 sao. Mình hài lòng về sản phẩm, mặc..."
2244,5,sản phẩm đúng như hình ảnh mô tả,0,0,0,0,1,Đánh giá 5 sao. sản phẩm đúng như hình ảnh mô tả
4691,3,"ao khoac nay may loi duog chi cai tui 1chut, c...",0,3,0,3,0,Đánh giá 3 sao. ao khoac nay may loi duog chi ...
5633,5,"Sp dày dặn hoàng hiện tốt ,cần thg để trải nghiệm",1,0,0,0,1,"Đánh giá 5 sao. Sp dày dặn hoàng hiện tốt ,cần..."
1739,5,chất lượng hơn giá tiền Ưng ý,0,0,0,0,1,Đánh giá 5 sao. chất lượng hơn giá tiền Ưng ý


In [117]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class ReviewDataset(Dataset):
    def __init__(self, texts, labels_df, tokenizer, max_len=256):
        self.texts = texts.values
        self.labels = labels_df.values # Chứa 5 cột nhãn
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        labels = self.labels[idx]

        # Tokenizer chuyển chữ thành số
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

# Khởi tạo Tokenizer của PhoBERT
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

# Tạo Dataset sử dụng quan_train và quan_test
train_dataset = ReviewDataset(df_train['model_input'], df_train[aspect_cols], tokenizer)
val_dataset = ReviewDataset(df_val['model_input'], df_val[aspect_cols], tokenizer)
test_dataset = ReviewDataset(df_test['model_input'], df_test[aspect_cols], tokenizer)

# Khởi tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Đã khởi tạo Dataset và DataLoader thành công.")

Đã khởi tạo Dataset và DataLoader thành công.


In [118]:
import torch.nn as nn
from transformers import AutoModel

class MultiOutputPhoBERT(nn.Module):
    def __init__(self, model_name="vinai/phobert-base-v2", n_classes=4):
        super().__init__()
        # Load phần Encoder của PhoBERT
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.5)

        hidden_size = self.encoder.config.hidden_size # 768

        # Xây dựng 5 lớp mạng Neural (Heads) để phân loại 5 khía cạnh
        self.head_chat_lieu = nn.Linear(hidden_size, n_classes)
        self.head_kich_co = nn.Linear(hidden_size, n_classes)
        self.head_thiet_ke = nn.Linear(hidden_size, n_classes)
        self.head_gia_cong = nn.Linear(hidden_size, n_classes)
        self.head_gia_tri = nn.Linear(hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # Lấy vector đại diện của toàn bộ câu (Token <s> đầu tiên)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)

        # Truyền vector vào 5 đầu ra độc lập
        out1 = self.head_chat_lieu(pooled_output)
        out2 = self.head_kich_co(pooled_output)
        out3 = self.head_thiet_ke(pooled_output)
        out4 = self.head_gia_cong(pooled_output)
        out5 = self.head_gia_tri(pooled_output)

        return out1, out2, out3, out4, out5

# Khởi tạo mô hình đưa lên GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiOutputPhoBERT().to(device)
print("Model đã sẵn sàng trên:", device)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model đã sẵn sàng trên: cuda


In [119]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
import torch.nn as nn

all_labels = []
for batch in train_loader:
    all_labels.append(batch['labels'].numpy())

all_labels = np.concatenate(all_labels)

criterion = []

for i in range(5):
    col_labels = all_labels[:, i]

    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1, 2, 3]),
        y=col_labels
    )

    weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
    criterion.append(nn.CrossEntropyLoss(weight=weights_tensor))

In [120]:
import torch
from torch.optim import AdamW
from tqdm import tqdm

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

optimizer = AdamW(model.parameters(), lr=2e-5)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    all_preds = []
    all_true = []

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=True)

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        preds = model(input_ids, attention_mask)

        loss = 0
        batch_preds = []

        for i in range(5):
            aspect_loss = criterion[i](preds[i], labels[:, i])
            loss += aspect_loss

            batch_preds.append(torch.argmax(preds[i], dim=1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        batch_preds = torch.stack(batch_preds, dim=1)

        all_preds.append(batch_preds.detach().cpu())
        all_true.append(labels.detach().cpu())

    avg_loss = total_loss / len(train_loader)

    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_true = torch.cat(all_true, dim=0).numpy()

    all_preds_flat = all_preds.flatten()
    all_true_flat = all_true.flatten()

    acc = accuracy_score(all_true_flat, all_preds_flat)
    precision = precision_score(all_true_flat, all_preds_flat, average='macro', zero_division=0)
    recall = recall_score(all_true_flat, all_preds_flat, average='macro', zero_division=0)
    f1 = f1_score(all_true_flat, all_preds_flat, average='macro', zero_division=0)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Acc: {acc:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Epoch 1/10:   0%|          | 0/99 [00:00<?, ?it/s]

Epoch 1/10: 100%|██████████| 99/99 [02:01<00:00,  1.22s/it, loss=6.3896]


Epoch 1/10 | Loss: 6.5096 | Acc: 0.4255 | Precision: 0.3563 | Recall: 0.4095 | F1: 0.3402


Epoch 2/10: 100%|██████████| 99/99 [02:07<00:00,  1.28s/it, loss=4.7946]


Epoch 2/10 | Loss: 5.5241 | Acc: 0.5516 | Precision: 0.4435 | Recall: 0.5448 | F1: 0.4472


...

Epoch 8/10 | Loss: 2.7650 | Acc: 0.8624 | Precision: 0.7471 | Recall: 0.8334 | F1: 0.7800


Epoch 9/10: 100%|██████████| 99/99 [02:13<00:00,  1.34s/it, loss=2.4599]


Epoch 9/10 | Loss: 2.4354 | Acc: 0.8872 | Precision: 0.7836 | Recall: 0.8658 | F1: 0.8159


Epoch 10/10: 100%|██████████| 99/99 [02:12<00:00,  1.34s/it, loss=2.1862]

Epoch 10/10 | Loss: 2.1595 | Acc: 0.9035 | Precision: 0.8107 | Recall: 0.8900 | F1: 0.8423


In [121]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

def evaluate_model(model, dataloader, device):
    model.eval() # Chuyển mô hình sang chế độ đánh giá (tắt Dropout)

    # Khởi tạo Dictionary lưu trữ dự đoán và nhãn thực tế cho 5 khía cạnh
    all_preds = {i: [] for i in range(5)}
    all_trues = {i: [] for i in range(5)}

    with torch.no_grad(): # Không tính đạo hàm để tiết kiệm bộ nhớ và tăng tốc
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device) # Kích thước (batch_size, 5)

            # Lấy dự đoán từ mô hình (Trả về 5 tensor cho 5 head)
            preds = model(input_ids, attention_mask)

            for i in range(5):
                # preds[i] có dạng (batch_size, 4). Dùng argmax để lấy vị trí có xác suất cao nhất
                pred_classes = torch.argmax(preds[i], dim=1).cpu().numpy()
                true_classes = labels[:, i].cpu().numpy()

                all_preds[i].extend(pred_classes)
                all_trues[i].extend(true_classes)

    return all_preds, all_trues

# --- TIẾN HÀNH ĐÁNH GIÁ ---
print("Đang đánh giá mô hình trên tập Val...")
all_preds, all_trues = evaluate_model(model, val_loader, device)

# Tên của 5 khía cạnh và 4 class để in ra cho đẹp
aspect_names = ['Chất Liệu', 'Kích Cỡ/Form', 'Thiết Kế', 'Gia Công', 'Giá Trị Thực Tế']
target_names = ["None", "Positive", "Negative", "Neutral"]

for i in range(5):
    print(f"\n{'='*50}")
    print(f"KHÍA CẠNH: {aspect_names[i].upper()}")
    print(f"{'='*50}")

    # 1. Tính Độ chính xác tổng thể (Accuracy)
    acc = accuracy_score(all_trues[i], all_preds[i])
    print(f"Accuracy: {acc:.4f}\n")

    # 2. In báo cáo chi tiết (Precision, Recall, F1-Score)
    # Gắn cứng labels=[0,1,2,3] để tránh lỗi nếu có class không xuất hiện trong test set
    report = classification_report(
        all_trues[i],
        all_preds[i],
        target_names=target_names,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
    print(report)


Đang đánh giá mô hình trên tập Val...

KHÍA CẠNH: CHẤT LIỆU
Accuracy: 0.8785

              precision    recall  f1-score   support

        None       0.96      0.95      0.95       265
    Positive       0.79      0.89      0.84        75
    Negative       0.67      0.69      0.68        32
     Neutral       0.44      0.30      0.36        23

    accuracy                           0.88       395
   macro avg       0.71      0.71      0.71       395
weighted avg       0.87      0.88      0.88       395


KHÍA CẠNH: KÍCH CỠ/FORM
Accuracy: 0.8557

              precision    recall  f1-score   support

        None       0.94      0.95      0.94       270
    Positive       0.79      0.70      0.74        53
    Negative       0.57      0.91      0.70        22
     Neutral       0.62      0.50      0.56        50

    accuracy                           0.86       395
   macro avg       0.73      0.76      0.74       395
weighted avg       0.86      0.86      0.85       395


KHÍA CẠN

In [124]:
def predict_review(rating, review_text, model, tokenizer, device):
    model.eval()

    # 1. Gộp chuỗi giống hệt như lúc Train
    input_text = f"Đánh giá {rating} sao. <s> {review_text}"

    # 2. Mã hóa văn bản
    encoding = tokenizer(
        input_text,
        truncation=True,
        max_length=256,
        padding='max_length',
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 3. Chạy mô hình
    with torch.no_grad():
        preds = model(input_ids, attention_mask)

    # 4. Ánh xạ số ngược lại thành chữ
    reverse_label_map = {0: "None", 1: "Positive", 2: "Negative", 3: "Neutral"}
    aspect_names = ['Chất Liệu', 'Kích Cỡ/Form', 'Thiết Kế', 'Gia Công', 'Giá Trị Thực Tế']

    print(f'Text Input: "{review_text}"')
    print(f'Rating: {rating} sao')
    print("-" * 35)
    print("KẾT QUẢ DỰ ĐOÁN:")

    for i in range(5):
        pred_class = torch.argmax(preds[i], dim=1).item()
        print(f" - {aspect_names[i]}: {reverse_label_map[pred_class]}")


# --- CHẠY THỬ ---
test_rating = 5
test_text = "quần kaki này chất vải rất tốt, cầm vào cảm giác dày dặn nhưng vẫn mềm, không thô hay cứng. Khi mặc lên đứng form, tôn dáng và không bị nhăn nhiều khi vận động. Màu áo không phai, giặt vài lần vẫn giữ được độ mới. Đường may chắc chắn, mép gấu và cạp xử lý gọn gàng, nhìn tổng thể rất chỉn chu. Dễ phối với áo thun, sơ mi hay hoodie, đi học, đi chơi đều ổn. Nói chung đây là chiếc áo xịn, đáng mua và dùng lâu vẫn đẹp."

predict_review(test_rating, test_text, model, tokenizer, device)


Text Input: "quần kaki này chất vải rất tốt, cầm vào cảm giác dày dặn nhưng vẫn mềm, không thô hay cứng. Khi mặc lên đứng form, tôn dáng và không bị nhăn nhiều khi vận động. Màu áo không phai, giặt vài lần vẫn giữ được độ mới. Đường may chắc chắn, mép gấu và cạp xử lý gọn gàng, nhìn tổng thể rất chỉn chu. Dễ phối với áo thun, sơ mi hay hoodie, đi học, đi chơi đều ổn. Nói chung đây là chiếc áo xịn, đáng mua và dùng lâu vẫn đẹp."
Rating: 5 sao
-----------------------------------
KẾT QUẢ DỰ ĐOÁN:
 - Chất Liệu: Positive
 - Kích Cỡ/Form: Positive
 - Thiết Kế: Neutral
 - Gia Công: Positive
 - Giá Trị Thực Tế: None


# Chuyển sang ONNX

In [130]:
!pip install -q -U onnx onnxruntime onnxscript


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 15.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.5 MB/s eta 0:00:00


In [135]:
import os
import json
import torch

EXPORT_DIR = path + "onnx_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(EXPORT_DIR, "multioutput_phobert_checkpoint.pth")
ONNX_PATH = os.path.join(EXPORT_DIR, "multioutput_phobert.onnx")
TOKENIZER_DIR = os.path.join(EXPORT_DIR, "tokenizer_phobert")

MAX_LEN = 256
MODEL_NAME = "vinai/phobert-base-v2"

id2label = {
    0: "None",
    1: "Positive",
    2: "Negative",
    3: "Neutral"
}

label2id = {
    "None": 0,
    "Positive": 1,
    "Negative": 2,
    "Neutral": 3
}

aspect_names = [
    "Chất Liệu",
    "Kích Cỡ/Form",
    "Thiết Kế",
    "Gia Công",
    "Giá Trị Thực Tế"
]

print("Export folder:", EXPORT_DIR)

Export folder: /content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export


In [136]:
model.eval()

torch.save({
    "model_state_dict": model.state_dict(),
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "aspect_cols": aspect_cols,
    "aspect_names": aspect_names,
    "id2label": id2label,
    "label2id": label2id
}, CHECKPOINT_PATH)

tokenizer.save_pretrained(TOKENIZER_DIR)

print("Đã lưu checkpoint:", CHECKPOINT_PATH)
print("Đã lưu tokenizer:", TOKENIZER_DIR)

Đã lưu checkpoint: /content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/multioutput_phobert_checkpoint.pth
Đã lưu tokenizer: /content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/tokenizer_phobert


In [137]:
import torch.nn as nn

class ONNXMultiOutputWrapper(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.trained_model = trained_model

    def forward(self, input_ids, attention_mask):
        out1, out2, out3, out4, out5 = self.trained_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = torch.stack([out1, out2, out3, out4, out5], dim=1)
        return logits

In [138]:
import torch
import numpy as np

model.to("cpu")
model.eval()

onnx_model = ONNXMultiOutputWrapper(model)
onnx_model.eval()

sample_text = "Đánh giá 5 sao. quần kaki chất vải tốt, form đẹp, đường may chắc chắn"

encoding = tokenizer(
    sample_text,
    truncation=True,
    max_length=MAX_LEN,
    padding="max_length",
    return_tensors="pt"
)

dummy_input_ids = encoding["input_ids"].long()
dummy_attention_mask = encoding["attention_mask"].long()

with torch.no_grad():
    torch_logits = onnx_model(dummy_input_ids, dummy_attention_mask)

print("Output PyTorch shape:", torch_logits.shape)

torch.onnx.export(
    onnx_model,
    args=(dummy_input_ids, dummy_attention_mask),
    f=ONNX_PATH,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size"},
        "attention_mask": {0: "batch_size"},
        "logits": {0: "batch_size"}
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False
)

print("Đã export ONNX:", ONNX_PATH)

Output PyTorch shape: torch.Size([1, 5, 4])


/tmp/ipykernel_10239/1196658084.py:28: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Đã export ONNX: /content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/multioutput_phobert.onnx


In [139]:
import onnx

onnx_check_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_check_model)

print("ONNX model hợp lệ!")

ONNX model hợp lệ!


In [140]:
import numpy as np
import onnxruntime as ort

ort_session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

onnx_inputs = {
    "input_ids": dummy_input_ids.numpy().astype(np.int64),
    "attention_mask": dummy_attention_mask.numpy().astype(np.int64)
}

onnx_logits = ort_session.run(["logits"], onnx_inputs)[0]

print("PyTorch logits shape:", torch_logits.detach().numpy().shape)
print("ONNX logits shape:", onnx_logits.shape)

max_diff = np.max(np.abs(torch_logits.detach().numpy() - onnx_logits))
print("Sai khác lớn nhất giữa PyTorch và ONNX:", max_diff)

PyTorch logits shape: (1, 5, 4)
ONNX logits shape: (1, 5, 4)
Sai khác lớn nhất giữa PyTorch và ONNX: 3.580004e-06


In [141]:
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

ort_session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

onnx_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)

def predict_review_onnx(rating, review_text):
    input_text = f"Đánh giá {rating} sao. {review_text}"

    encoding = onnx_tokenizer(
        input_text,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        return_tensors="np"
    )

    onnx_inputs = {
        "input_ids": encoding["input_ids"].astype(np.int64),
        "attention_mask": encoding["attention_mask"].astype(np.int64)
    }

    logits = ort_session.run(["logits"], onnx_inputs)[0]
    pred_ids = logits.argmax(axis=-1)[0]

    result = {}
    for aspect, pred_id in zip(aspect_names, pred_ids):
        result[aspect] = id2label[int(pred_id)]

    return result

In [142]:
test_rating = 5
test_text = "quần kaki này chất vải rất tốt, mặc đứng form, đường may chắc chắn"

result = predict_review_onnx(test_rating, test_text)
result

{'Chất Liệu': 'Positive',
 'Kích Cỡ/Form': 'Positive',
 'Thiết Kế': 'None',
 'Gia Công': 'Positive',
 'Giá Trị Thực Tế': 'None'}

In [143]:
ZIP_PATH = path + "onnx_export.zip"

!zip -r "{ZIP_PATH}" "{EXPORT_DIR}"

print("Đã tạo file zip:", ZIP_PATH)

  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/ (stored 0%)
  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/multioutput_phobert_checkpoint.pth (deflated 7%)
  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/tokenizer_phobert/ (stored 0%)
  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/tokenizer_phobert/tokenizer_config.json (deflated 76%)
  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/tokenizer_phobert/added_tokens.json (stored 0%)
  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/tokenizer_phobert/vocab.txt (deflated 55%)
  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/tokenizer_phobert/bpe.codes (deflated 59%)
  adding: content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export/multioutput_phobert.onnx (deflated 7%)
Đã tạo file zip: /content/drive/MyDrive/KhaiPhaDL/BTL/onnx_export.zip
